# 04 · Evaluate  (the money shot)

Four conditions, all four reported. **Never quote the in-domain number
alone** — it is the most flattering and the least informative.

| Condition | Purpose | Expectation |
|---|---|---|
| in-domain | comparability with published work | strong |
| out-of-domain (In-the-Wild) | honest generalisation | materially worse |
| codec-degraded | validates the thesis | augmented holds, baseline collapses |
| leave-one-attack-out | zero-day generalisation | the number that reflects deployment |

A team that shows where its model fails, and can explain why, reads as the
team that understood the problem.

In [ ]:
# --- Colab setup -----------------------------------------------------------
# Run this first in every notebook.  Idempotent.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    # Keep the repo and all caches on Drive so a disconnect does not cost you
    # the feature extraction pass.
    PROJECT = Path("/content/drive/MyDrive/voice-integrity")
    if not PROJECT.exists():
        raise SystemExit(
            f"Upload or clone the repo to {PROJECT} first.\n"
            "  !git clone <your-repo-url> /content/drive/MyDrive/voice-integrity"
        )
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

# Repo-local model cache.  Set BEFORE importing transformers, or it will use
# the default location and the cache will not be portable to the demo machine.
os.environ["HF_HOME"] = str(PROJECT / "cache" / "huggingface")
os.environ["TORCH_HOME"] = str(PROJECT / "cache" / "torch")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

print("project:", PROJECT)
print("python :", sys.version.split()[0])

In [ ]:
if IN_COLAB:
    !pip install -q transformers speechbrain soundfile librosa pydantic pyyaml cryptography wandb
    !apt-get -qq install -y ffmpeg libopencore-amrnb-dev > /dev/null

# AMR-NB encoding is the one that silently goes missing.  If this prints
# nothing, your mobile-codec augmentation does nothing and the whole
# codec-robustness result quietly evaporates.
!ffmpeg -hide_banner -encoders 2>/dev/null | grep -i amr || echo "AMR-NB ENCODER MISSING"

In [ ]:
import torch
print("cuda available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device         :", torch.cuda.get_device_name(0))
    print("memory         : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from vif.common.config import load_config
from vif.data.manifests import read_manifest
from vif.models.heads import load_checkpoint

config = load_config("configs")

models = {}
for name, path in [
    ("baseline",     "models/checkpoints/baseline.pt"),
    ("codec_robust", "models/checkpoints/codec_robust.pt"),
]:
    head, meta = load_checkpoint(
        path, config.model.head,
        feat_dim=config.model.frontend.hidden_dim,
        expect_window=config.model.audio.window_samples,
        expect_frontend=config.model.frontend.model_id,
    )
    models[name] = head.to(DEVICE).eval()
    print(f"{name:<14} epoch {meta.get('epoch')}  dev EER "
          f"{meta.get('metrics', {}).get('eer', float('nan'))*100:.2f}%")

In [ ]:
from vif.eval.runner import EvalReport, score_arrays
from vif.train.loop import score_manifest

report = EvalReport()

conditions = {
    "in-domain (clean)":    ("data/manifests/asvspoof19la_eval.jsonl", "data/features/eval_clean"),
    "in-domain (codec)":    ("data/manifests/asvspoof19la_eval.jsonl", "data/features/eval_codec"),
    "out-of-domain (wild)": ("data/manifests/in_the_wild_eval.jsonl",  "data/features/wild_clean"),
}

for condition, (manifest, features) in conditions.items():
    if not Path(features).exists():
        print(f"skip {condition}: extract features first")
        continue
    items = read_manifest(manifest)
    for name, head in models.items():
        scores = score_manifest(head, items, features, device=DEVICE)
        # Align by manifest index: an item with no cached features is skipped.
        labels, aligned = score_arrays(items, scores)
        report.add(condition, name, labels, aligned)

print()
print(report.table())

## The chart that is the presentation

Two models, same audio, clean versus codec-degraded. Eight seconds of
explanation, and it proves the one thing nobody else in the room addressed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = ["clean", "codec-degraded"]
baseline = [
    next((r.metrics.eer*100 for r in report.results
          if r.model == "baseline" and "clean" in r.name), np.nan),
    next((r.metrics.eer*100 for r in report.results
          if r.model == "baseline" and "codec" in r.name), np.nan),
]
robust = [
    next((r.metrics.eer*100 for r in report.results
          if r.model == "codec_robust" and "clean" in r.name), np.nan),
    next((r.metrics.eer*100 for r in report.results
          if r.model == "codec_robust" and "codec" in r.name), np.nan),
]

x = np.arange(2); width = 0.36
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar(x - width/2, baseline, width, label="baseline (clean training)", color="#B8873C")
ax.bar(x + width/2, robust,   width, label="codec-augmented training", color="#0C7180")
ax.axhline(50, ls=":", c="grey", lw=1)
ax.text(1.45, 50.6, "chance", color="grey", fontsize=9, ha="right")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("EER  (%, lower is better)")
ax.set_title("What the phone network does to a detector")
ax.legend(); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig("data/codec_robustness.png", dpi=200); plt.show()

## Leave-one-attack-out

Each generator evaluated on its own against the full bonafide set. This is the
number that reflects deployment against a synthesizer the model has never
seen, and almost nobody reports it.

In [ ]:
from vif.eval.runner import leave_one_attack_out, per_condition

items = read_manifest("data/manifests/asvspoof19la_eval.jsonl")
scores = score_manifest(models["codec_robust"], items, "data/features/eval_clean", device=DEVICE)

per_attack = leave_one_attack_out(items, scores)
worst = sorted(per_attack.items(), key=lambda kv: -kv[1].eer)[:5]
print("\nhardest attacks for this model:")
for attack, metrics in worst:
    print(f"  {attack:<8} {metrics.summary()}")

In [ ]:
# Per-codec breakdown, straight from the condition labels the augmenter wrote.
codec_items = read_manifest("data/features/eval_codec/manifest.jsonl")
codec_scores = score_manifest(models["codec_robust"], codec_items,
                              "data/features/eval_codec", device=DEVICE)
_ = per_condition(codec_items, codec_scores)

In [ ]:
report.save("data/eval_report.json")
print(report.table())
print()
from vif.eval.runner import compare_models
print(compare_models(report, "in-domain (codec)"))

## Say this out loud

Report the poor out-of-domain number rather than only the flattering
in-domain one. Prepared weaknesses read as maturity; discovered weaknesses
read as sloppiness. Same facts, opposite impression.